# Executive Hotel Booking EDA

**Dataset:** Day 15 Executive Hotel Booking Dataset

## Objective
Perform an end-to-end exploratory data analysis covering data quality, cleaning, descriptive statistics, univariate and bivariate analysis, group-wise comparisons, correlation analysis, visualization, business insights, and management recommendations.

> **Important:** The cleaning decisions are documented rather than silently changing the source data. Duplicate Booking IDs are removed for analysis; categorical inconsistencies are standardized; missing operational fields are handled explicitly; and outliers are assessed using the IQR method.


## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

file_name = "Day15_Executive_Hotel_Booking_EDA_Dataset.csv"
df_raw = pd.read_csv(file_name)

print("Raw shape:", df_raw.shape)
display(df_raw.head())


## 2. Understand Structure and Data Quality

In [ ]:
print("Shape:", df_raw.shape)
print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df_raw.isnull().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", df_raw.duplicated().sum())
print("Duplicate Booking IDs:", df_raw["Booking_ID"].duplicated().sum())

print("\nUnique values in selected categorical fields:")
for col in ["Hotel_Type","Market_Segment","Meal_Type","Hotel_Location","Reservation_Status"]:
    print(f"\n{col}:")
    print(df_raw[col].value_counts(dropna=False).head(15))


## 3. Data Cleaning and Preprocessing

In [ ]:
df = df_raw.copy()

# Standardize text/categorical values
df["Hotel_Type"] = df["Hotel_Type"].astype(str).str.strip().str.title()
df["Market_Segment"] = df["Market_Segment"].astype(str).str.strip().str.title()
df["Hotel_Location"] = df["Hotel_Location"].astype(str).str.strip().str.title()

df["Meal_Type"] = (
    df["Meal_Type"].astype(str).str.strip()
      .replace({"bb":"BB", "B&B":"BB", "B & B":"BB"})
      .str.upper()
      .replace({"UNDEFINED":"Undefined", "UNKNOWN":"Unknown"})
)

# Convert dates
for col in ["Booking_Date","Arrival_Date","Reservation_Status_Date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Handle missing values with documented choices
df["Country"] = df["Country"].fillna("Unknown")
df["Agent_ID"] = df["Agent_ID"].fillna(0)       # 0 = no agent recorded
df["Company_ID"] = df["Company_ID"].fillna(0)   # 0 = no company recorded
df["Children"] = df["Children"].fillna(0)
df["ADR"] = df["ADR"].fillna(df["ADR"].median())
df["Satisfaction_Score"] = df["Satisfaction_Score"].fillna(df["Satisfaction_Score"].median())

# Remove duplicate booking IDs, retaining the first record
before = len(df)
df = df.drop_duplicates(subset="Booking_ID", keep="first").copy()

# Derived analysis fields
df["Booking_Month"] = df["Booking_Date"].dt.to_period("M").astype(str)
df["Arrival_Month"] = df["Arrival_Date"].dt.to_period("M").astype(str)
df["Lead_Time_Band"] = pd.cut(
    df["Lead_Time_Days"], [-1,7,30,90,180,365,10000],
    labels=["0-7","8-30","31-90","91-180","181-365","366+"]
)
df["Revenue_Per_Night"] = df["Estimated_Revenue"] / df["Total_Nights"].replace(0, np.nan)

print("Rows before duplicate-ID removal:", before)
print("Rows after cleaning:", len(df))
print("Remaining missing values:", df.isnull().sum().sum())


### Cleaning decisions
- **Duplicate Booking IDs:** removed for analysis because Booking_ID should identify a unique booking.
- **Text inconsistencies:** whitespace and case variants were standardized.
- **Dates:** converted to datetime.
- **Agent/Company missingness:** encoded as `0`, representing no recorded agent/company rather than inventing an ID.
- **Children:** missing values treated as 0.
- **ADR and Satisfaction Score:** median imputation used because these are continuous measures and only a small number of records are missing.
- **Country:** missing values labeled `Unknown`.


## 4. Outlier Assessment Using IQR

In [ ]:
outlier_cols = ["Lead_Time_Days","ADR","Estimated_Revenue","Total_Nights"]

outlier_summary = []
for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append([col, q1, q3, lower, upper, int(count)])

outlier_table = pd.DataFrame(
    outlier_summary,
    columns=["Variable","Q1","Q3","Lower Bound","Upper Bound","Outlier Count"]
)
display(outlier_table.round(2))


**Interpretation:** The IQR method identifies unusually high or low observations without automatically deleting them. Hotel bookings can legitimately have long lead times, high ADR, or long stays, so the analysis preserves these observations and treats them as potentially meaningful business cases rather than assuming they are errors.

## 5. Descriptive Statistics

In [ ]:
numeric_cols = [
    "Lead_Time_Days","Weekend_Nights","Weekday_Nights","Adults","Children",
    "Babies","Previous_Cancellations","Previous_Bookings","Booking_Changes",
    "Days_In_Waiting_List","Total_Nights","ADR","Required_Car_Parking_Spaces",
    "Total_Special_Requests","Satisfaction_Score","Estimated_Revenue"
]
display(df[numeric_cols].describe().T.round(2))


## 6. Univariate Analysis — Cancellation Rate

In [ ]:
status_counts = df["Reservation_Status"].value_counts()
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="Reservation_Status", order=status_counts.index)
plt.title("Reservation Status Distribution")
plt.xlabel("Reservation Status")
plt.ylabel("Number of Bookings")
plt.tight_layout()
plt.show()

print("Cancellation rate:", round(df["Is_Canceled"].mean()*100, 2), "%")


**Interpretation:** The dataset has a high cancellation rate of approximately **72.2%**, making cancellation management a central revenue-protection opportunity.

## 7. Revenue and ADR Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

sns.histplot(df["ADR"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Average Daily Rate (ADR)")
axes[0].set_xlabel("ADR")
axes[0].set_ylabel("Frequency")

sns.histplot(df["Estimated_Revenue"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Distribution of Estimated Revenue")
axes[1].set_xlabel("Estimated Revenue")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


**Interpretation:** ADR and estimated revenue show substantial variation across bookings. The right tails indicate a smaller number of higher-value bookings, supporting differentiated pricing and revenue-management strategies.

## 8. Monthly Booking Trend — Line Plot

In [ ]:
monthly = df.groupby("Booking_Month").agg(
    Bookings=("Booking_ID","count"),
    Revenue=("Estimated_Revenue","sum")
).reset_index()

plt.figure(figsize=(13,5))
sns.lineplot(data=monthly, x="Booking_Month", y="Bookings", marker="o")
plt.title("Monthly Booking Volume")
plt.xlabel("Booking Month")
plt.ylabel("Number of Bookings")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


**Interpretation:** The monthly line plot reveals changes in booking demand over time. Management can use recurring peaks and troughs to plan staffing, inventory, promotions, and pricing.

## 9. Revenue by Hotel Type — Bar Chart

In [ ]:
hotel_summary = df.groupby("Hotel_Type").agg(
    Bookings=("Booking_ID","count"),
    Revenue=("Estimated_Revenue","sum"),
    Avg_ADR=("ADR","mean"),
    Cancellation_Rate=("Is_Canceled","mean")
).sort_values("Revenue", ascending=False)

display(hotel_summary.round(3))

plt.figure(figsize=(8,5))
sns.barplot(data=hotel_summary.reset_index(), x="Hotel_Type", y="Revenue")
plt.title("Estimated Revenue by Hotel Type")
plt.xlabel("Hotel Type")
plt.ylabel("Estimated Revenue")
plt.tight_layout()
plt.show()


**Interpretation:** **Resort Hotel** generates the most estimated revenue in the dataset. Revenue should be interpreted alongside cancellation rate and ADR rather than bookings alone.

## 10. Revenue by Hotel Location — Bar Chart

In [ ]:
location_summary = df.groupby("Hotel_Location").agg(
    Bookings=("Booking_ID","count"),
    Revenue=("Estimated_Revenue","sum"),
    Avg_ADR=("ADR","mean")
).sort_values("Revenue", ascending=False)

display(location_summary.round(2))

plt.figure(figsize=(12,5))
sns.barplot(data=location_summary.reset_index(), x="Hotel_Location", y="Revenue")
plt.title("Estimated Revenue by Hotel Location")
plt.xlabel("Hotel Location")
plt.ylabel("Estimated Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Interpretation:** **Goa** is the highest-revenue location. Location-level differences can guide where management prioritizes inventory, marketing, and service investments.

## 11. Market Segment Performance — Grouped Analysis

In [ ]:
segment_summary = df.groupby("Market_Segment").agg(
    Bookings=("Booking_ID","count"),
    Revenue=("Estimated_Revenue","sum"),
    Avg_ADR=("ADR","mean"),
    Cancellation_Rate=("Is_Canceled","mean"),
    Avg_Lead_Time=("Lead_Time_Days","mean")
).sort_values("Revenue", ascending=False)

display(segment_summary.round(2))


**Interpretation:** **Online Ta** contributes the highest total revenue. Segment-level comparison is useful because high booking volume does not necessarily imply the highest value per booking or the lowest cancellation risk.

## 12. Lead Time vs Cancellation — Box Plot

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df, x="Is_Canceled", y="Lead_Time_Days")
plt.title("Lead Time by Cancellation Status")
plt.xlabel("Canceled (0 = No, 1 = Yes)")
plt.ylabel("Lead Time (Days)")
plt.tight_layout()
plt.show()


**Interpretation:** Longer lead times are associated with cancellation behavior in this dataset (correlation with cancellation flag ≈ **0.34**). This suggests that long-lead bookings should be monitored for cancellation risk and supported by appropriate confirmation or deposit policies.

## 13. ADR vs Estimated Revenue — Scatter Plot

In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(data=df, x="ADR", y="Estimated_Revenue", alpha=0.5)
sns.regplot(data=df, x="ADR", y="Estimated_Revenue", scatter=False, ci=None)
plt.title("ADR vs Estimated Revenue")
plt.xlabel("Average Daily Rate (ADR)")
plt.ylabel("Estimated Revenue")
plt.tight_layout()
plt.show()

print("Correlation:", round(df["ADR"].corr(df["Estimated_Revenue"]), 3))


**Interpretation:** ADR has a positive relationship with estimated revenue (correlation ≈ **0.60**). Pricing and length of stay therefore jointly contribute to booking value.

## 14. Customer Type vs Cancellation — Bar Chart

In [ ]:
customer_cancel = df.groupby("Customer_Type")["Is_Canceled"].mean().sort_values(ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(x=customer_cancel.index, y=customer_cancel.values * 100)
plt.title("Cancellation Rate by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Cancellation Rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

display((customer_cancel*100).round(2).to_frame("Cancellation Rate (%)"))


**Interpretation:** Cancellation risk differs across customer types. Management can use these differences to tailor deposit requirements, communication, and retention strategies.

## 15. Room Type Assignment — Count Plot

In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(data=df, x="Room_Type_Assigned",
              order=df["Room_Type_Assigned"].value_counts().index)
plt.title("Assigned Room Type Distribution")
plt.xlabel("Assigned Room Type")
plt.ylabel("Bookings")
plt.tight_layout()
plt.show()


**Interpretation:** The distribution highlights which room types are most frequently assigned. This can support inventory planning and reveal where room demand is concentrated.

## 16. Special Requests vs Satisfaction — Violin Plot

In [ ]:
plt.figure(figsize=(10,5))
sns.violinplot(data=df, x="Total_Special_Requests", y="Satisfaction_Score", inner="quartile")
plt.title("Customer Satisfaction by Number of Special Requests")
plt.xlabel("Total Special Requests")
plt.ylabel("Satisfaction Score")
plt.tight_layout()
plt.show()


**Interpretation:** Satisfaction distributions vary with the number of special requests. This can help management understand whether fulfilling additional guest needs is associated with stronger customer experience.

## 17. Correlation Heatmap

In [ ]:
corr_cols = [
    "Lead_Time_Days","Weekend_Nights","Weekday_Nights","Adults","Children",
    "Previous_Cancellations","Previous_Bookings","Booking_Changes",
    "Days_In_Waiting_List","Total_Nights","ADR",
    "Required_Car_Parking_Spaces","Total_Special_Requests",
    "Satisfaction_Score","Is_Canceled","Estimated_Revenue"
]

corr = df[corr_cols].corr()

plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, linewidths=0.3)
plt.title("Correlation Heatmap — Hotel Booking Variables")
plt.tight_layout()
plt.show()


**Interpretation:** The heatmap provides a high-level view of linear relationships. Revenue is influenced by ADR and stay-related variables, while cancellation is associated with booking-behavior variables such as lead time. Correlation does not establish causation, so operational decisions should be validated with further analysis.

## 18. Executive KPI Summary

In [ ]:
kpis = pd.Series({
    "Bookings after deduplication": len(df),
    "Cancellation rate (%)": df["Is_Canceled"].mean()*100,
    "Total estimated revenue": df["Estimated_Revenue"].sum(),
    "Average ADR": df["ADR"].mean(),
    "Average total nights": df["Total_Nights"].mean(),
    "Repeat guest rate (%)": df["Is_Repeated_Guest"].mean()*100,
    "Average satisfaction score": df["Satisfaction_Score"].mean()
})
display(kpis.to_frame("Value").round(2))


# 19. Executive Findings

### 1. Cancellation is a major revenue risk
The cancellation rate is high, so revenue protection should be a management priority. Long-lead bookings show higher cancellation exposure, making them useful candidates for targeted confirmation and deposit policies.

### 2. Revenue is concentrated across hotel types and locations
The strongest hotel type and location contribute disproportionately to estimated revenue. Management should compare these markets on ADR, cancellation, and booking volume before allocating additional resources.

### 3. Market segments have different economics
Segments differ in booking volume, ADR, lead time, and cancellation behavior. A single commercial strategy is therefore unlikely to be optimal across all segments.

### 4. Pricing and stay length are important revenue drivers
Higher ADR is positively associated with estimated revenue. Revenue management should consider both price and length of stay rather than focusing only on occupancy or booking count.

### 5. Customer experience is a strategic metric
Satisfaction varies across guest behaviors such as special requests. Monitoring satisfaction alongside booking and revenue KPIs can help management protect repeat business and service quality.


# 20. Management Recommendations

1. **Strengthen cancellation controls:** Use deposits, flexible-but-tiered cancellation policies, and proactive confirmation for high-risk long-lead bookings.
2. **Adopt segment-specific commercial strategies:** Set different pricing, promotion, and cancellation policies for Direct, OTA, Corporate, Groups, and other segments based on their economics.
3. **Prioritize high-value markets:** Focus marketing and inventory investment on the strongest hotel types and locations while diagnosing weaker markets.
4. **Improve revenue management:** Use ADR, stay length, seasonality, and segment demand together to optimize rates and packages.
5. **Monitor booking-channel economics:** Evaluate revenue and cancellation performance by distribution channel, not just booking volume, to control acquisition costs and risk.
6. **Use customer-experience KPIs operationally:** Track satisfaction and special-request fulfillment alongside revenue to identify service improvements that can support loyalty.
7. **Build an executive dashboard:** Refresh monthly KPIs for bookings, cancellations, ADR, estimated revenue, lead time, segment mix, and satisfaction to support faster management decisions.


# 21. Final Conclusion

The EDA demonstrates that hotel booking performance is shaped by **cancellation risk, location, hotel type, market segment, pricing, stay behavior, and customer experience**. The most actionable priorities are to reduce avoidable cancellations, optimize pricing by segment and market, and monitor customer experience alongside financial performance.

The analysis preserves potentially legitimate extreme bookings while documenting their presence, avoids treating missing agent/company identifiers as real entities, and standardizes inconsistent categorical values before analysis. This provides a cleaner and more defensible foundation for management decisions.


# 22. Submission Checklist

- Run every notebook cell from top to bottom.
- Confirm all visualizations render correctly.
- Save the notebook as `.ipynb`.
- Upload the notebook to a public GitHub repository.
- Verify the notebook opens correctly from GitHub.
- Submit the GitHub link in the LMS.
- Submit the accompanying executive EDA report.
